# Exploratory Data Analysis -- Classroom Reaction Recognition

This notebook provides a visual overview of the annotated dataset used to train and evaluate the ResNet18 and VGG16 reaction classifiers.

In [ ]:
import random
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

PROJECT_ROOT = Path("..")
CROPS_DIR = PROJECT_ROOT / "data" / "crops"
SPLITS = ["train", "val", "test"]
CLASSES = ["Neutral", "Confused", "Smiling_Amused", "Surprised", "Bored_Tired"]
SEED = 42

## 1. Class Distribution Across Splits

We count the number of annotated images per class in each split to understand the data balance.

In [ ]:
counts = defaultdict(dict)
for split in SPLITS:
    for cls in CLASSES:
        folder = CROPS_DIR / split / cls
        n = len(list(folder.glob("*.jpg"))) if folder.exists() else 0
        counts[split][cls] = n

x = np.arange(len(SPLITS))
width = 0.15

fig, ax = plt.subplots(figsize=(12, 5))
for i, cls in enumerate(CLASSES):
    vals = [counts[s][cls] for s in SPLITS]
    bars = ax.bar(x + i * width, vals, width, label=cls)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                str(v), ha="center", va="bottom", fontsize=8, fontweight="bold")

ax.set_xticks(x + width * 2)
ax.set_xticklabels(SPLITS, fontsize=12)
ax.set_ylabel("Number of Samples", fontsize=12)
ax.set_title("Reaction Class Distribution Across Train / Val / Test Splits", fontsize=14, fontweight="bold")
ax.legend(title="Reaction Class", fontsize=9)
plt.tight_layout()
plt.show()

### Observations

The training set may be **imbalanced** across the five reaction classes. Some reactions (e.g., `Surprised`) are naturally rarer in classroom footage, while `Neutral` tends to dominate.

This imbalance justifies our use of **Weighted `CrossEntropyLoss`** during training, where each class weight is computed as `total_samples / (num_classes * class_count)`. Without this, the model would trivially predict the majority class and still achieve a misleadingly high accuracy.

## 2. Sample Images From Each Training Class

A grid showing three randomly selected crops per reaction class from the **training** split. This helps verify annotation quality and understand visual differences between reactions.

In [ ]:
random.seed(SEED)

num_classes = len(CLASSES)
fig, axes = plt.subplots(num_classes, 3, figsize=(10, 3.5 * num_classes))

for row, cls in enumerate(CLASSES):
    folder = CROPS_DIR / "train" / cls
    images = sorted(folder.glob("*.jpg"))
    samples = random.sample(images, min(3, len(images))) if images else []

    for col in range(3):
        ax = axes[row][col]
        if col < len(samples):
            img = Image.open(samples[col])
            ax.imshow(img)
            ax.set_title(samples[col].name, fontsize=7)
        ax.axis("off")

    axes[row][0].set_ylabel(cls, fontsize=11, rotation=0, labelpad=100, va="center")

fig.suptitle("Training Set -- Sample Images per Reaction Class", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 3. Dataset Limitations

Key constraints to acknowledge:

1. **Small dataset size.** The entire annotated corpus may be limited in scale. Production reaction recognition systems would require thousands of samples with inter-annotator agreement.

2. **Single annotator.** All labels were assigned by one person, introducing potential subjective bias in distinguishing between reaction categories (e.g., *Confused* vs *Neutral*).

3. **Limited camera viewpoints.** Training data comes from classroom videos with fixed camera angles. Variations in lighting, seating density, or camera perspective may impact generalization.

4. **Class imbalance.** Some reaction classes (e.g., `Surprised`) are naturally rare in classroom footage. Per-class metrics (especially Recall) should be interpreted with caution for underrepresented classes.

5. **Facial expression ambiguity.** Facial reactions are inherently subjective and context-dependent. A furrowed brow could indicate concentration or confusion depending on context.